# Model A Final Evaluation: Locked Enriched Logistic Regression

## Purpose

This notebook performs the **one-time final future-holdout evaluation** for the Model A configuration selected and checkpointed in notebook 104:

- Model: Logistic Regression (`max_iter=1000`)
- Feature set: Enriched 17 initial-purchase predictors
- Development boundary: `first_order_date <= 2018-04-24 09:10:20`
- Final holdout: `first_order_date > 2018-04-24 09:10:20`

The configuration was selected from development-period temporal validation before this notebook evaluates final-holdout performance. This notebook does not reconsider the model, features, preprocessing, hyperparameters, or threshold after seeing final results.

### How to read this evaluation

This notebook is the **one-time final future-holdout evaluation** of the configuration locked in notebook 104: Enriched 17 + Logistic Regression. It is not a model-selection or tuning exercise; all preprocessing is fitted on the development period before the holdout is transformed and scored.

## Locked configuration and guardrails

The selected feature set contains the original eight prediction-time-safe fields plus nine enriched initial-purchase fields. The target is `repeat_purchase_90d`; time is `first_order_date`.

No tuning, class weighting, resampling, SMOTE, threshold optimization, or additional feature engineering is performed. The final holdout is evaluated once only after all imputers, indicators, scalers, category-frequency rules, encoders, and model coefficients have been fit using the development period alone.

In [1]:
# Import the fixed modeling stack and construct a portable, credential-free database connection.
# The connection reads the SQL-built feature table; this notebook does not create or alter database objects.
import math
import os
from getpass import getuser

import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# DATABASE_URL makes the notebooks portable without storing a password or machine-specific user name.
# If it is unset, use the documented local PostgreSQL convention for the active operating-system user.
database_url = os.getenv(
    "DATABASE_URL",
    f"postgresql+psycopg2://{getuser()}@localhost:5432/olist_project",
)
engine = create_engine(database_url)

## Load the locked population and report final-holdout labels

Final-holdout labels are intentionally read here because the model-selection checkpoint has already been committed. The SQL table is read only; no database object is created or modified.

In [2]:
# Recreate the pre-locked Enriched 17 feature list and the fixed time boundary from notebook 104.
# The full data load is now appropriate because the configuration was selected before this one-time holdout evaluation.
id_column = "customer_unique_id"
target_column = "repeat_purchase_90d"
time_column = "first_order_date"
final_holdout_boundary = pd.Timestamp("2018-04-24 09:10:20")

original_features = [
    "customer_state",
    "first_order_amount",
    "freight_to_order_ratio",
    "products_ordered",
    "unique_products_ordered",
    "number_of_categories",
    "number_of_sellers",
    "payment_installments",
]
new_features = [
    "primary_category",
    "payment_type_group",
    "payment_record_count",
    "first_order_month",
    "first_order_weekday",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "any_seller_same_state",
    "avg_customer_seller_distance_km",
]
enriched_features = original_features + new_features
assert len(original_features) == 8
assert len(enriched_features) == 17

selected_columns = [id_column, target_column, time_column] + enriched_features
model_data = pd.read_sql(
    f"""
    SELECT {', '.join(selected_columns)}
    FROM customer_initial_purchase_model_enriched
    ORDER BY first_order_date, customer_unique_id
    """,
    engine,
    parse_dates=[time_column],
)
assert model_data[id_column].is_unique
assert set(model_data[target_column].unique()) == {"No", "Yes"}

development = model_data.loc[model_data[time_column] <= final_holdout_boundary].copy()
final_holdout = model_data.loc[model_data[time_column] > final_holdout_boundary].copy()
assert development[time_column].max() <= final_holdout_boundary
assert development[time_column].max() < final_holdout[time_column].min()
assert set(development[id_column]).isdisjoint(set(final_holdout[id_column]))

def population_summary(name, frame):
    positives = int((frame[target_column] == "Yes").sum())
    return {
        "period": name,
        "rows": len(frame),
        "positives": positives,
        "negatives": len(frame) - positives,
        "positive_prevalence": positives / len(frame),
        "minimum_first_order_date": frame[time_column].min(),
        "maximum_first_order_date": frame[time_column].max(),
    }

population = pd.DataFrame([
    population_summary("Development", development),
    population_summary("Final holdout", final_holdout),
])
display(population)

,period,rows,positives,negatives,positive_prevalence,minimum_first_order_date,maximum_first_order_date
0,Development,69637,1480,68157,0.0213,2016-09-04 21:15:19,2018-04-24 09:10:20
1,Final holdout,17287,227,17060,0.0131,2018-04-24 09:16:29,2018-07-19 17:24:35


## Development-fitted preprocessing

This repeats the selected notebook-104 strategy exactly:

- Numeric values: median imputation and missingness indicators, then standardization.
- Categorical missing values: deterministic `missing` token.
- Categories: training-fitted `OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=25)`.

The full development period is the training data for every learned preprocessing component. The final holdout is transformed only by those fitted components.

In [3]:
# Keep all learned preprocessing inside a Pipeline. Medians, missingness indicators, scaling parameters,
# category frequencies, and one-hot columns are learned from development rows only, never the future holdout.
categorical_features = [
    "customer_state",
    "primary_category",
    "payment_type_group",
    "first_order_month",
    "first_order_weekday",
    "any_seller_same_state",
]
assert set(categorical_features).issubset(enriched_features)

def prepare_feature_frame(frame):
    """Apply only deterministic category type/missing handling before fitted preprocessing."""
    X = frame[enriched_features].copy()
    for column in categorical_features:
        X[column] = X[column].astype("object").where(X[column].notna(), "missing").astype(str)
    return X

def make_preprocessor():
    numeric_features = [column for column in enriched_features if column not in categorical_features]
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=25,
        sparse_output=False,
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        sparse_threshold=0,
    )

X_development = prepare_feature_frame(development)
X_holdout = prepare_feature_frame(final_holdout)
y_development = (development[target_column] == "Yes").astype(int)
y_holdout = (final_holdout[target_column] == "Yes").astype(int)

assert list(X_development.columns) == enriched_features
assert list(X_holdout.columns) == enriched_features
assert len(X_development) + len(X_holdout) == len(model_data)

## Train the locked model once

The pipeline is fitted once on the complete development period. Its Logistic Regression configuration is unchanged from the selected temporal-validation experiment.

In [4]:
# Fit the locked Enriched 17 Logistic Regression exactly once on all development-period customers.
# This is not a new selection, tuning, resampling, weighting, or threshold-optimization step.
final_model = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(max_iter=1000)),
])
final_model.fit(X_development, y_development)

# All transformations above were fitted by Pipeline.fit using development data only.
assert final_model.named_steps["model"].__class__.__name__ == "LogisticRegression"

## Final metrics and ranking metrics

Accuracy is reported for completeness but is not decisive with a rare target. Average Precision is primary; ROC-AUC provides complementary ranking evidence. Top-K metrics evaluate the business-style question: among customers ranked highest by repeat probability, how concentrated are actual repeat purchasers relative to the overall period prevalence? They do not optimize the 0.50 classification threshold.

In [5]:
# Evaluate both default-threshold classification and probability ranking. Top-K metrics target the most
# highly ranked customers without changing the fixed 0.50 classification threshold; lift compares their
# repeat rate with overall final-holdout prevalence.
def ranking_metrics(y_true, probabilities, percentage):
    y_array = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    k = max(1, math.ceil(len(y_array) * percentage))
    top_indices = np.argsort(-probabilities, kind="mergesort")[:k]
    captured = int(y_array[top_indices].sum())
    prevalence = float(y_array.mean())
    precision_at_k = captured / k
    recall_at_k = captured / int(y_array.sum()) if y_array.sum() else np.nan
    lift_at_k = precision_at_k / prevalence if prevalence else np.nan
    return {
        "customers": k,
        "repeat_purchasers_captured": captured,
        "precision": precision_at_k,
        "recall": recall_at_k,
        "lift": lift_at_k,
    }

def evaluate_predictions(y_true, probabilities):
    predictions = (np.asarray(probabilities) >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),
        "positive_prevalence": float(np.mean(y_true)),
        "predicted_positives_at_050": int(predictions.sum()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

development_probabilities = final_model.predict_proba(X_development)[:, 1]
holdout_probabilities = final_model.predict_proba(X_holdout)[:, 1]

development_metrics = evaluate_predictions(y_development, development_probabilities)
holdout_metrics = evaluate_predictions(y_holdout, holdout_probabilities)
metrics_table = pd.DataFrame(
    [development_metrics, holdout_metrics],
    index=["Development", "Final holdout"],
)
display(metrics_table)

print("Development confusion matrix [[TN, FP], [FN, TP]]:")
print([[development_metrics[key] for key in ("tn", "fp")], [development_metrics[key] for key in ("fn", "tp")]])
print("\nFinal-holdout confusion matrix [[TN, FP], [FN, TP]]:")
print([[holdout_metrics[key] for key in ("tn", "fp")], [holdout_metrics[key] for key in ("fn", "tp")]])
print(f"\nDevelopment-to-holdout ROC-AUC gap: {development_metrics['roc_auc'] - holdout_metrics['roc_auc']:.4f}")
print(f"Development-to-holdout AP gap: {development_metrics['average_precision'] - holdout_metrics['average_precision']:.4f}")

top_1pct = ranking_metrics(y_holdout, holdout_probabilities, 0.01)
top_5pct = ranking_metrics(y_holdout, holdout_probabilities, 0.05)
ranking_table = pd.DataFrame([top_1pct, top_5pct], index=["Top 1%", "Top 5%"])
display(ranking_table)

,accuracy,precision,recall,f1,roc_auc,average_precision,positive_prevalence,predicted_positives_at_050,tn,fp,fn,tp
Development,0.9787,0.0000,0.0000,0.0000,0.6501,0.0415,0.0213,0,68157,0,1480,0
Final holdout,0.9869,0.0000,0.0000,0.0000,0.5440,0.0186,0.0131,0,17060,0,227,0


Development confusion matrix [[TN, FP], [FN, TP]]:
[[68157, 0], [1480, 0]]

Final-holdout confusion matrix [[TN, FP], [FN, TP]]:
[[17060, 0], [227, 0]]

Development-to-holdout ROC-AUC gap: 0.1061
Development-to-holdout AP gap: 0.0229


,customers,repeat_purchasers_captured,precision,recall,lift
Top 1%,173,7,0.0405,0.0308,3.0814
Top 5%,865,19,0.0220,0.0837,1.6728


## Compare final performance with development-only temporal validation

Notebook 104 selected this configuration using development folds only:

| Reference | AP | ROC-AUC | Top-1% lift | Top-5% lift |
|---|---:|---:|---:|---:|
| Mean validation | 0.0290 | 0.5785 | 2.4082 | 2.0402 |
| Fold range | 0.0274–0.0318 | 0.5571–0.5989 | 2.1706–2.6277 | 1.6713–2.6390 |

The comparison below describes whether the one-time final future period is consistent with those pre-holdout expectations. It does not trigger any model revision.

In [6]:
# Compare the one-time future result with the locked development-only temporal-validation reference.
# This provides context for generalization without reopening model selection after seeing the holdout.
temporal_validation_reference = {
    "mean_ap": 0.0290,
    "ap_min": 0.0274,
    "ap_max": 0.0318,
    "mean_roc_auc": 0.5785,
    "roc_auc_min": 0.5571,
    "roc_auc_max": 0.5989,
    "mean_top_1pct_lift": 2.4082,
    "mean_top_5pct_lift": 2.0402,
}
comparison_to_validation = pd.DataFrame([{
    "metric": "Average Precision",
    "temporal_validation_mean": temporal_validation_reference["mean_ap"],
    "final_holdout": holdout_metrics["average_precision"],
    "final_minus_validation_mean": holdout_metrics["average_precision"] - temporal_validation_reference["mean_ap"],
}, {
    "metric": "ROC-AUC",
    "temporal_validation_mean": temporal_validation_reference["mean_roc_auc"],
    "final_holdout": holdout_metrics["roc_auc"],
    "final_minus_validation_mean": holdout_metrics["roc_auc"] - temporal_validation_reference["mean_roc_auc"],
}, {
    "metric": "Top-1% lift",
    "temporal_validation_mean": temporal_validation_reference["mean_top_1pct_lift"],
    "final_holdout": top_1pct["lift"],
    "final_minus_validation_mean": top_1pct["lift"] - temporal_validation_reference["mean_top_1pct_lift"],
}, {
    "metric": "Top-5% lift",
    "temporal_validation_mean": temporal_validation_reference["mean_top_5pct_lift"],
    "final_holdout": top_5pct["lift"],
    "final_minus_validation_mean": top_5pct["lift"] - temporal_validation_reference["mean_top_5pct_lift"],
}])
display(comparison_to_validation)

print("Final AP within the development-fold range:", temporal_validation_reference["ap_min"] <= holdout_metrics["average_precision"] <= temporal_validation_reference["ap_max"])
print("Final ROC-AUC within the development-fold range:", temporal_validation_reference["roc_auc_min"] <= holdout_metrics["roc_auc"] <= temporal_validation_reference["roc_auc_max"])

,metric,temporal_validation_mean,final_holdout,final_minus_validation_mean
0,Average Precision,0.0290,0.0186,-0.0104
1,ROC-AUC,0.5785,0.5440,-0.0345
2,Top-1% lift,2.4082,3.0814,0.6732
3,Top-5% lift,2.0402,1.6728,-0.3674


Final AP within the development-fold range: False
Final ROC-AUC within the development-fold range: False


## Coefficient associations

Coefficients describe associations with the model’s predicted log-odds after the selected preprocessing. They are not causal effects. Numeric coefficients are on standardized, imputed scales; one-hot coefficients are relative to the encoded reference structure and can be affected by correlated basket features. Missingness-indicator coefficients describe model associations with absent metadata, not customer behavior.

In [7]:
# Coefficients are associations within this fitted, encoded model—not causal effects. One-hot coefficients
# are interpreted relative to omitted/reference categories, and numeric effects use standardized inputs.
transformed_feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.DataFrame({
    "transformed_feature": transformed_feature_names,
    "coefficient": final_model.named_steps["model"].coef_[0],
})

print("Strongest positive associations with predicted repeat probability:")
display(coefficients.nlargest(12, "coefficient").reset_index(drop=True))
print("\nStrongest negative associations with predicted repeat probability:")
display(coefficients.nsmallest(12, "coefficient").reset_index(drop=True))

Strongest positive associations with predicted repeat probability:


,transformed_feature,coefficient
0,categorical__primary_category_home_appliances,1.1436
1,categorical__primary_category_fashion_bags_acc...,0.6012
2,categorical__primary_category_home_confort,0.5318
3,categorical__primary_category_bed_bath_table,0.5172
4,categorical__primary_category_furniture_bedroom,0.5014
5,categorical__primary_category_furniture_decor,0.3623
6,categorical__primary_category_sports_leisure,0.3171
7,categorical__primary_category_air_conditioning,0.2928
8,categorical__customer_state_MT,0.2912
9,categorical__customer_state_RO,0.2547



Strongest negative associations with predicted repeat probability:


,transformed_feature,coefficient
0,categorical__primary_category_cool_stuff,-0.9176
1,categorical__any_seller_same_state_False,-0.8829
2,categorical__any_seller_same_state_True,-0.8789
3,categorical__primary_category_electronics,-0.6780
4,categorical__primary_category_consoles_games,-0.5393
5,categorical__payment_type_group_debit_card,-0.5215
6,categorical__payment_type_group_credit_card_pl...,-0.4770
7,categorical__primary_category_luggage_accessories,-0.4722
8,categorical__payment_type_group_credit_card,-0.4711
9,categorical__customer_state_SC,-0.4294


## Final Model A conclusion

This is the one-time final-holdout evaluation of the configuration locked after temporal validation. The outputs above answer whether initial-purchase information provides future ranking signal, whether enrichment generalized, and whether a top-ranked targeting list concentrates actual repeat purchasers.

The result should be interpreted as a portfolio analysis rather than a production deployment: repeat purchase is rare, the Olist observation period changes over time, default-threshold classification is not the same as ranking utility, and the model uses only available initial-purchase information. Future work should retain the locked final result, then investigate threshold policy, calibration, business targeting costs, and any subsequent model design only through a new time-aware development process.

In [8]:
# State the final result honestly: rare-event accuracy and a 0.50 classifier are not evidence of practical
# utility when no positives are predicted. The locked configuration is not changed after this comparison.
ap_vs_prevalence = holdout_metrics["average_precision"] / holdout_metrics["positive_prevalence"]
if holdout_metrics["average_precision"] >= temporal_validation_reference["ap_min"]:
    signal_statement = "The modest development-period ranking signal persisted into the final future holdout."
elif holdout_metrics["average_precision"] > holdout_metrics["positive_prevalence"]:
    signal_statement = "Ranking signal persisted but weakened relative to the development-period validation range."
else:
    signal_statement = "The development-period ranking signal did not persist beyond holdout prevalence."

print(signal_statement)
print(
    f"Final Average Precision is {ap_vs_prevalence:.2f} times the final-holdout repeat prevalence; "
    f"top-1% lift is {top_1pct['lift']:.2f} and top-5% lift is {top_5pct['lift']:.2f}."
)
print("The model and feature set remain locked; these final results do not trigger further selection or tuning.")

Ranking signal persisted but weakened relative to the development-period validation range.
Final Average Precision is 1.41 times the final-holdout repeat prevalence; top-1% lift is 3.08 and top-5% lift is 1.67.
The model and feature set remain locked; these final results do not trigger further selection or tuning.
